# Влияние параметра C в SVC

В этой работе проверяется, как параметр `C` влияет на классификацию автомобилей.

## 1. Загрузка данных

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

RANDOM_STATE = 42
data_path = Path("car_dataset.csv")
if not data_path.exists():
    data_path = Path("src") / "car_dataset.csv"
if not data_path.exists():
    raise FileNotFoundError("Не найден файл src/car_dataset.csv")

data = pd.read_csv(data_path)
target_column = "segment"
feature_columns = [column for column in data.columns if column != target_column]
X = data[feature_columns]
y = data[target_column]

print(f"Размер данных: {data.shape[0]} автомобилей")
print(f"Количество признаков: {len(feature_columns)}")
print(y.value_counts())

## 2. Разбиение данных и параметры

Маленькое значение `C` сильнее допускает ошибки на обучении. Большое значение `C` заставляет модель точнее разделять обучающие объекты, но может привести к переобучению.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

kernels = ["linear", "rbf", "poly", "sigmoid"]
c_values = 10.0 ** np.arange(-3, 4)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1_macro": "f1_macro",
}

print("Обучающая выборка:", X_train.shape)
print("Тестовая выборка:", X_test.shape)

## 3. Перебор C и ядер

In [ ]:
rows = []
for kernel in kernels:
    for c_value in c_values:
        model = Pipeline(
            [
                ("scaler", StandardScaler()),
                ("svc", SVC(C=c_value, kernel=kernel)),
            ]
        )
        started = time.perf_counter()
        scores = cross_validate(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring=scoring,
            n_jobs=1,
            return_train_score=True,
        )
        rows.append(
            {
                "kernel": kernel,
                "C": c_value,
                "accuracy": scores["test_accuracy"].mean(),
                "balanced_accuracy": scores["test_balanced_accuracy"].mean(),
                "f1_macro": scores["test_f1_macro"].mean(),
                "train_accuracy": scores["train_accuracy"].mean(),
                "fit_time": scores["fit_time"].mean(),
                "elapsed_time": time.perf_counter() - started,
            }
        )

results = pd.DataFrame(rows)
results.sort_values("balanced_accuracy", ascending=False).head(10)

## 4. Графики

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
for kernel in kernels:
    part = results[results["kernel"] == kernel]
    axes[0].plot(part["C"], part["balanced_accuracy"], marker="o", label=kernel)
    gap = part["train_accuracy"] - part["accuracy"]
    axes[1].plot(part["C"], gap, marker="o", label=kernel)

for axis in axes:
    axis.set_xscale("log")
    axis.set_xlabel("C")
    axis.grid(alpha=0.3)
    axis.legend()

axes[0].set_title("Balanced accuracy в CV")
axes[0].set_ylabel("Balanced accuracy")
axes[1].set_title("Разрыв между train и validation")
axes[1].set_ylabel("Train accuracy - validation accuracy")
plt.show()

## 5. Проверка лучшей модели на тесте

In [ ]:
best_row = results.loc[results["balanced_accuracy"].idxmax()]
best_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("svc", SVC(C=float(best_row["C"]), kernel=best_row["kernel"])),
    ]
)
best_model.fit(X_train, y_train)
test_prediction = best_model.predict(X_test)

print(f"Лучшее ядро: {best_row['kernel']}")
print(f"Лучшее C: {best_row['C']}")
print(f"Test accuracy: {accuracy_score(y_test, test_prediction):.4f}")
print(f"Test balanced accuracy: {balanced_accuracy_score(y_test, test_prediction):.4f}")
print(f"Test F1 macro: {f1_score(y_test, test_prediction, average='macro'):.4f}")

## Вывод

По таблице и графикам можно увидеть, при каких значениях `C` качество растёт, а при каких появляется заметное переобучение.